# 01 — GARCH(1,1) (weekly silver volatility)

GARCH(1,1) is the classic parametric benchmark for conditional volatility. Unlike
HAR-RV — which regresses *realised* volatility on its own lags — GARCH models the
volatility of the **return series itself**, so it provides an econometric reference
point of a different kind.

The weekly return series and the RV target both come from `volatility_weekly.csv`,
built in `00_features.ipynb` — run that notebook first.


## Setup


In [1]:
import sys, os
sys.path.append(os.path.abspath('../../src'))

import numpy as np
import pandas as pd
from arch import arch_model
from vol_utils import vol_evaluate, vol_period_metrics, vol_diebold_mariano
from eval_utils import PERIODS
import warnings; warnings.filterwarnings('ignore')

frame = pd.read_csv('../../data/processed/volatility_weekly.csv',
                    parse_dates=['Date']).set_index('Date')
test_df = frame[frame['split'] == 'test']

y_test     = test_df['target'].values
prev_test  = test_df['rv_w_lag1'].values
weekly_ret = frame['silver_ret']               # weekly silver log-returns, full sample
print(f'test weeks: {len(test_df)}')
print(y_test[:5])
print(prev_test[:5])
print(weekly_ret[:5])


test weeks: 175
[0.03579865 0.03010313 0.02336985 0.02631828 0.05362404]
[0.02768859 0.03579865 0.03010313 0.02336985 0.02631828]
Date
2015-04-03   -0.021697
2015-04-10   -0.019120
2015-04-17   -0.009144
2015-04-24   -0.037051
2015-05-01    0.030246
Name: silver_ret, dtype: float64


## 1. Walk-forward GARCH(1,1)

The model assumes

$$r_t = \mu + \varepsilon_t, \qquad \varepsilon_t = \sigma_t z_t, \qquad
\sigma_t^2 = \omega + \alpha\,\varepsilon_{t-1}^2 + \beta\,\sigma_{t-1}^2.$$

The one-step-ahead conditional volatility $\hat\sigma_t$ is used as the prediction for
$\text{RV}_t$. To avoid look-ahead the model is **re-fit walking forward week by
week**: at each test week the GARCH is estimated only on weekly returns strictly prior
to that week, then forecast one step ahead. Returns are scaled by 100 during fitting
for numerical stability, and the forecast is scaled back afterwards.


In [2]:
garch_preds = []
for d in test_df.index:
    hist = weekly_ret.loc[:d].iloc[:-1]                            # returns strictly before week d
    m   = arch_model(hist * 100, mean='Constant', vol='GARCH', p=1, q=1, dist='normal')
    res = m.fit(disp='off', show_warning=False)
    fc  = res.forecast(horizon=1, reindex=False)
    garch_preds.append(np.sqrt(fc.variance.values[-1, 0]) / 100)   # back to return scale
garch_preds = np.array(garch_preds)
print(f'fitted {len(garch_preds)} walk-forward GARCH models')


fitted 175 walk-forward GARCH models


## 2. What `arch_model.fit()` actually does

The loop above calls `.fit()` 175 times. Each call hides quite a lot — this section
unpacks what one fit does, why $\sigma_t$ is **latent**, and how the four parameters
$(\mu, \omega, \alpha, \beta)$ are pinned down. The cell below re-runs a single
non-walk-forward fit on train+val so the printed `summary()` shows the estimates each
loop iteration in §1 produces.

### The latent-volatility setup

Split the model into two pieces:

$$
\underbrace{r_t = \mu + \varepsilon_t,\quad \varepsilon_t = \sigma_t z_t,\ z_t \sim \mathcal{N}(0, 1)}_{\text{observed equation (returns)}}
$$
$$
\underbrace{\sigma_t^2 = \omega + \alpha\,\varepsilon_{t-1}^2 + \beta\,\sigma_{t-1}^2}_{\text{latent equation (variance)}}
$$

Only $r_t$ is observed. The conditional variance $\sigma_t^2$ is a **latent state** —
nowhere in the data does it appear as a column you can plot against a forecast. All we
have is the recursion that ties it to past **squared shocks** $\varepsilon_{t-1}^2 =
(r_{t-1} - \mu)^2$ and past **variances** $\sigma_{t-1}^2$. Crucially the recursion is
deterministic once you fix the four parameters and one starting value
$\sigma_0^2$, so given $(\mu, \omega, \alpha, \beta)$ we can compute $\sigma_t^2$ for
every $t$ purely from the observed $r_t$. The whole variance path is a function of the
parameters.

### Maximum-likelihood estimation

Conditioning on the past, $r_t \mid \mathcal{F}_{t-1} \sim \mathcal{N}(\mu, \sigma_t^2)$
under `dist='normal'`. The conditional log-likelihood of the sample is

$$
\ell(\mu, \omega, \alpha, \beta) \;=\; -\tfrac{n}{2}\log(2\pi)
\;-\; \tfrac{1}{2}\sum_{t=1}^{n}\!\left[\log \sigma_t^2 \;+\; \tfrac{(r_t - \mu)^2}{\sigma_t^2}\right]
$$

with each $\sigma_t^2$ supplied by the latent recursion above. Maximising $\ell$
jointly over the four parameters is what `arch_model.fit()` does:

- starts from sensible initial values (sample mean and variance, $\alpha + \beta$
  near 0.9);
- runs a numerical optimiser (SLSQP by default in the `arch` package) with
  **constraints** $\omega > 0$, $\alpha, \beta \ge 0$ and $\alpha + \beta < 1$ to keep
  the unconditional variance $\bar\sigma^2 = \omega / (1 - \alpha - \beta)$ finite;
- returns standard errors from the Hessian at the optimum.

The variance series is then **filtered**: given the MLE parameters, run the recursion
forward to recover the most-likely $\hat\sigma_t$ at every $t$. That filtered series is
what `.forecast()` extrapolates one step ahead in §1 — `arch` returns the variance, we
square-root it to get the volatility forecast.

### Reading the parameters

- $\mu$ — the constant conditional mean of weekly returns (close to 0 here).
- $\omega$ — the floor of the variance equation; together with $\alpha + \beta$ it sets
  the long-run unconditional variance $\omega / (1 - \alpha - \beta)$.
- $\alpha$ — the *shock* coefficient. How much last week's squared innovation
  contributes to this week's variance. High $\alpha$ → spiky / reactive volatility.
- $\beta$ — the *persistence* coefficient. How much last week's variance carries over.
  High $\beta$ → smoother, slowly-decaying volatility.
- $\alpha + \beta$ — the **total persistence**. Values close to 1 mean volatility
  shocks decay slowly (long memory); the **half-life of a shock** is
  $\log(0.5) / \log(\alpha + \beta)$ weeks. Values $\ge 1$ would imply the
  unconditional variance is infinite — the constraint $\alpha + \beta < 1$ is what
  keeps the model stationary.

### Why "latent" matters for evaluation

Because $\sigma_t$ is never observed, we can **never directly check** how good the
$\hat\sigma_t$ estimate is. Every volatility evaluation in this chapter uses a *proxy*
— weekly realised volatility built from daily squared returns in `00_features` — and
Patton's (2011) proxy-robust **QLIKE** loss is what lets the chapter still rank
competing $\hat\sigma_t$ series consistently across noisy realisations of the proxy.
See [`notes.md`](notes.md) for the math.


In [3]:
# One-off fit on train+val (no walk-forward) so we can inspect the parameters that
# each iteration of section 1's loop is actually estimating.
hist0    = weekly_ret.loc[:test_df.index[0]].iloc[:-1]     # everything strictly before the first test week
demo_fit = arch_model(hist0 * 100, mean='Constant', vol='GARCH', p=1, q=1,
                      dist='normal').fit(disp='off', show_warning=False)
print(demo_fit.summary())

mu  = demo_fit.params['mu']
om  = demo_fit.params['omega']
al  = demo_fit.params['alpha[1]']
be  = demo_fit.params['beta[1]']
persistence = al + be
half_life   = np.log(0.5) / np.log(persistence) if 0 < persistence < 1 else np.inf
long_run_var = om / (1 - persistence) if persistence < 1 else np.nan

print(f'\n  alpha + beta (persistence)       = {persistence:.4f}    '
      f'(stationarity requires < 1)')
print(f'  half-life of a vol shock         = {half_life:.1f} weeks')
print(f'  long-run variance omega/(1-a-b)  = {long_run_var:.4f}  '
      f'(on the *100-scaled returns; sqrt -> {np.sqrt(long_run_var)/100:.4f} weekly vol)')


                     Constant Mean - GARCH Model Results                      
Dep. Variable:             silver_ret   R-squared:                       0.000
Mean Model:             Constant Mean   Adj. R-squared:                  0.000
Vol Model:                      GARCH   Log-Likelihood:               -1076.04
Distribution:                  Normal   AIC:                           2160.09
Method:            Maximum Likelihood   BIC:                           2176.10
                                        No. Observations:                  405
Date:                Sun, May 24 2026   Df Residuals:                      404
Time:                        17:43:15   Df Model:                            1
                               Mean Model                               
                 coef    std err          t      P>|t|  95.0% Conf. Int.
------------------------------------------------------------------------
mu            -0.0978      0.168     -0.584      0.559 [ -0.426,  0.23

## 3. Evaluation


In [4]:
results = [vol_evaluate('GARCH(1,1)', y_test, garch_preds, prev_test)]


GARCH(1,1)                      RMSE=0.03296  MAE=0.01824  R2=+0.249  DCA=0.674


## 4. DM vs the Naïve floor — does GARCH(1,1) genuinely beat $\text{RV}_{t-1}$?

GARCH models the volatility of the *return* series, so unlike HAR-RV its prediction is
a parametric conditional standard deviation rather than a regression on past RV. The
test below asks the same headline question — is this prediction lower-loss than the
Naïve $\text{RV}_{t-1}$ floor under Diebold-Mariano (1995) with Newey-West (1987) lag-1
variance via `vol_diebold_mariano`? Negative DM = GARCH has lower loss.

**QLIKE is the primary loss.** Weekly silver RV is heavy-tailed enough that under
squared-error loss a handful of extreme weeks dominate the differential and inflate the
DM variance, so an RMSE improvement that is real and steady can still fail an MSE-DM
test. The volatility-forecasting literature (Patton 2011) reports forecasts under
**QLIKE**, a proxy-robust ratio loss; squared-error DM is kept underneath only as a
reference. The same comparison is collected across models in `evaluation.ipynb` §4.


In [5]:
# Diebold-Mariano: GARCH(1,1) vs the Naive RV_{t-1} floor.
print('QLIKE loss  --  primary test:')
vol_diebold_mariano(y_test, garch_preds, prev_test, 'GARCH(1,1)', 'Naive', loss='qlike')

print('\nSquared-error loss  --  reference:')
vol_diebold_mariano(y_test, garch_preds, prev_test, 'GARCH(1,1)', 'Naive', loss='mse');


QLIKE loss  --  primary test:
GARCH(1,1)                   vs Naive         [qlike]  DM=-2.594  p=0.009  **    -> winner: GARCH(1,1)

Squared-error loss  --  reference:
GARCH(1,1)                   vs Naive         [mse  ]  DM=-1.161  p=0.246  (ns)  -> winner: tie


## 5. Sub-period breakdown

RMSE and DCA split by calendar year, using the shared `PERIODS` definition.


In [6]:
period_garch = vol_period_metrics(y_test, garch_preds, prev_test, test_df.index, PERIODS)
period_garch.to_csv('../../data/processed/period_garch_volatility.csv')
period_garch.round(4)


,n,RMSE,MAE,DCA
Period,,,,
2023 (choppy),52,0.0162,0.0140,0.6731
2024 (bull start),52,0.0145,0.0122,0.7885
2025 (bull run),52,0.0251,0.0195,0.6346
2026 (YTD),19,0.0836,0.0431,0.4737
── Full test ──,175,0.0330,0.0182,0.6743


## 6. Save outputs

- `metrics_garch_volatility.csv` — GARCH(1,1) headline metrics
- `pred_garch_volatility.csv` — test-set predictions, consumed by `evaluation.ipynb`


In [7]:
pd.DataFrame(results).to_csv('../../data/processed/metrics_garch_volatility.csv', index=False)

pred_garch = pd.DataFrame({'actual': y_test, 'prev': prev_test, 'garch': garch_preds},
                          index=test_df.index)
pred_garch.to_csv('../../data/processed/pred_garch_volatility.csv', index_label='Date')
print('Saved metrics_garch_volatility.csv + pred_garch_volatility.csv')
pd.DataFrame(results).round(5)


Saved metrics_garch_volatility.csv + pred_garch_volatility.csv


,model,rmse,mae,r2,dca
0,"GARCH(1,1)",0.03296,0.01824,0.24899,0.67429
